# Kavach on Evon v3.3 (Q4_K_M, Kaggle 2× T4): v3.1 pipeline on 24 test calls

Loads `anilkumarlucky/gnani-evon-v3.3-30b-a3b-GGUF` (Q4_K_M) into `llama-server`, split across both T4s, and answers three questions for Kavach:

1. **Does it fit?** Full GPU offload first; if a T4 runs out of memory, the first N layers' MoE experts move to CPU RAM (`--n-cpu-moe`) and we retry.
2. **How fast?** Prompt (prefill) and generation tokens/s, one request and two at once (Prosecutor + Defender run in parallel).
3. **Do the Kavach roles work?** v3 pipeline (checklist Prosecutor, verification in code, rule verdict, Evon explanation and Telugu warning) on 12 synthetic scam calls and 12 genuine calls, thinking off.

**Before Run all**
1. Settings → Accelerator: **GPU T4 ×2**. Settings → Internet: **on**.
2. Add-ons → Secrets → add `HF` = your Hugging Face token, and tick it for this notebook.

Everything is written to `/kaggle/working/evon_t4_report_v3.md` (+ `.json`) at the end; download that and bring it back.

In [ ]:
# 1. Settings
import os, json, time, subprocess, shutil, re, threading
from pathlib import Path

GGUF_REPO  = "anilkumarlucky/gnani-evon-v3.3-30b-a3b-GGUF"
GGUF_FILE  = "gnani-evon-v3.3-30b-a3b-Q4_K_M.gguf"
BASE_REPO  = "gnani/gnani-evon-v3.3-30b-a3b"      # chat_template.jinja lives here, not in tokenizer_config
LLAMA_SHA  = "b3fa699"                             # the llama.cpp commit the GGUF was converted with
SRC        = Path("/tmp/llama.cpp")
BIN        = Path("/kaggle/working/llama-bin")     # keep binaries so a rerun in the same session skips the build
MODEL_DIR  = Path("/tmp/evon")                     # /kaggle/working is capped at 20 GB, so the model goes to /tmp
PORT       = 8080
URL        = f"http://127.0.0.1:{PORT}"
CTX, SLOTS = 16384, 2                              # 8k tokens per slot, two slots for parallel roles
CPU_MOE_STEPS = [0, 6, 12, 20]                     # retry ladder if the T4s run out of memory

REPORT = {"started": time.strftime("%Y-%m-%d %H:%M:%S")}
def note(k, v):
    REPORT[k] = v
    print(f"{k}: {v}")

In [ ]:
# 2. Hardware check
!nvidia-smi --query-gpu=index,name,memory.total,memory.used --format=csv
!nvcc --version | tail -2
!free -g | head -2; df -h /tmp /kaggle/working | tail -2
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.split("\n")
gpus = [g for g in gpus if g.strip()]
note("gpus", gpus)
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"

## Build llama-server with CUDA (about 15–25 min, once per session)

Only the T4's architecture (sm_75) is compiled, which cuts the build time a lot. Static linking keeps the binary self-contained. Kaggle ships no CUDA stub library, so CMake is pointed at the real driver in /usr/local/nvidia/lib64.

In [ ]:
# 3. Build
SERVER = BIN / "llama-server"
if not SERVER.exists():
    t = time.time()
    if not SRC.exists():
        !git clone -q --filter=blob:none https://github.com/ggml-org/llama.cpp {SRC}
    !git -C {SRC} checkout -q {LLAMA_SHA}
    !cd {SRC} && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DBUILD_SHARED_LIBS=OFF -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_cuda_driver_LIBRARY=/usr/local/nvidia/lib64/libcuda.so > /tmp/cmake.log 2>&1 || tail -30 /tmp/cmake.log
    !cd {SRC} && cmake --build build --target llama-server -j {os.cpu_count()} > /tmp/build.log 2>&1 || tail -40 /tmp/build.log
    BIN.mkdir(parents=True, exist_ok=True)
    shutil.copy(SRC / "build/bin/llama-server", SERVER)
    note("build_minutes", round((time.time() - t) / 60, 1))
assert SERVER.exists(), "build failed, see the log tail above"
note("llama_cpp", LLAMA_SHA)

In [ ]:
# 4. Download the Q4_K_M GGUF (private repo) and Evon's chat template
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF")
t = time.time()
MODEL = Path(hf_hub_download(GGUF_REPO, GGUF_FILE, local_dir=MODEL_DIR))
TEMPLATE = Path(hf_hub_download(BASE_REPO, "chat_template.jinja", local_dir=MODEL_DIR))
note("model_gb", round(MODEL.stat().st_size / 1e9, 2))
note("download_minutes", round((time.time() - t) / 60, 1))

## Start the server

`-sm layer -ts 1,1` puts half the layers on each T4. If loading fails with an out-of-memory error, the next step of `CPU_MOE_STEPS` keeps that many layers' experts in CPU RAM (slower, but only 6 of 128 experts are active per token).

In [ ]:
# 5. Start llama-server, stepping down the GPU load until it fits
import requests

def gpu_mem():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
                         capture_output=True, text=True).stdout.strip().splitlines()
    return [f"{int(u)/1024:.1f}/{int(t)/1024:.1f} GB" for u, t in (l.split(", ") for l in out)]

def start_server(n_cpu_moe):
    log = open(f"/tmp/server_cpumoe{n_cpu_moe}.log", "w")
    cmd = [str(SERVER), "-m", str(MODEL), "--chat-template-file", str(TEMPLATE),
           "-ngl", "999", "-sm", "layer", "-ts", "1,1", "-c", str(CTX), "-np", str(SLOTS),
           "-fa", "auto", "--host", "127.0.0.1", "--port", str(PORT), "--metrics"]
    if n_cpu_moe:
        cmd += ["--n-cpu-moe", str(n_cpu_moe)]
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t = time.time()
    while time.time() - t < 900:
        if proc.poll() is not None:
            return None, log.name
        try:
            if requests.get(f"{URL}/health", timeout=2).status_code == 200:
                return proc, log.name
        except requests.RequestException:
            pass
        time.sleep(5)
    proc.kill()
    return None, log.name

server = None
for n in CPU_MOE_STEPS:
    t = time.time()
    server, logname = start_server(n)
    if server:
        note("n_cpu_moe", n)
        note("load_seconds", round(time.time() - t))
        note("vram_after_load", gpu_mem())
        break
    tail = open(logname).read()[-1500:]
    print(f"--- n_cpu_moe={n} failed; log tail:\n{tail}")
    if not re.search(r"out of memory|failed to allocate|cudaMalloc", tail, re.I):
        raise SystemExit("Server failed for a reason other than memory; see the log tail above.")
assert server, "Did not fit even with the largest --n-cpu-moe step"

In [ ]:
# 6. Client helper. Thinking is on by default in Evon's template, so every call says which it wants.
def chat(messages, max_tokens=512, thinking=False, schema=None, temperature=0.3, cache_prompt=True):
    body = {"messages": messages, "max_tokens": max_tokens, "temperature": temperature,
            "chat_template_kwargs": {"enable_thinking": thinking}, "cache_prompt": cache_prompt}
    if schema:
        body["response_format"] = {"type": "json_object", "schema": schema}
    t = time.time()
    r = requests.post(f"{URL}/v1/chat/completions", json=body, timeout=900)
    r.raise_for_status()
    j = r.json()
    msg = j["choices"][0]["message"]
    tm = j.get("timings", {})
    return {"content": msg.get("content") or "", "reasoning": msg.get("reasoning_content") or "",
            "wall_s": round(time.time() - t, 2),
            "prompt_tokens": tm.get("prompt_n"), "prompt_tps": round(tm.get("prompt_per_second", 0), 1),
            "gen_tokens": tm.get("predicted_n"), "gen_tps": round(tm.get("predicted_per_second", 0), 1),
            "finish": j["choices"][0].get("finish_reason")}

def n_tokens(text):
    return len(requests.post(f"{URL}/tokenize", json={"content": text}).json()["tokens"])

warm = chat([{"role": "user", "content": "Say OK."}], max_tokens=8)
print(warm)

## Speed

In [ ]:
# 7. Generation and prefill speed, then two requests at once
from concurrent.futures import ThreadPoolExecutor

speed = {}
r = chat([{"role": "user", "content": "Write 200 words about why people should never share an OTP on a phone call."}],
         max_tokens=300, temperature=0.7, cache_prompt=False)
speed["single_gen"] = {k: r[k] for k in ("prompt_tokens", "prompt_tps", "gen_tokens", "gen_tps", "wall_s")}

long_text = ("A bank customer receives a call. The caller claims to be from the cyber police and says a parcel "
             "in the customer's name contained drugs. ") * 120
r = chat([{"role": "user", "content": long_text + "\nSummarise the above in one sentence."}],
         max_tokens=40, cache_prompt=False)
speed["prefill_long"] = {k: r[k] for k in ("prompt_tokens", "prompt_tps", "gen_tokens", "gen_tps", "wall_s")}

prompts = ["List five signs of a phone scam.", "List five signs that a bank call is genuine."]
t = time.time()
with ThreadPoolExecutor(2) as ex:
    rs = list(ex.map(lambda p: chat([{"role": "user", "content": p}], max_tokens=200, temperature=0.7,
                                    cache_prompt=False), prompts))
speed["two_parallel"] = {"wall_s": round(time.time() - t, 2), "per_request_gen_tps": [x["gen_tps"] for x in rs],
                         "total_gen_tokens": sum(x["gen_tokens"] for x in rs)}
speed["two_parallel"]["aggregate_gen_tps"] = round(speed["two_parallel"]["total_gen_tokens"] / speed["two_parallel"]["wall_s"], 1)

te = "మీ ఆధార్ నంబర్‌తో ఒక పార్సెల్‌లో డ్రగ్స్ దొరికాయి. ఈ విషయం ఎవరికీ చెప్పకూడదు, వెంటనే వీడియో కాల్‌లో ఉండండి."
speed["telugu_tokens_per_word"] = round(n_tokens(te) / len(te.split()), 2)
note("speed", speed)
note("vram_after_speed", gpu_mem())

## Kavach roles, v3 (thinking off throughout)

What v2 showed: grammar-forced JSON with an evidence *list* made Evon always fill the list. It tagged harmless lines ("come fasting for the blood test") as a secrecy demand, and the Judge treated any evidence as proof, so all 14 calls came out SCAM.

v3 splits the work:

1. **Prosecutor = checklist.** One yes/no per tactic, with the caller's exact words for every yes. "No" is the easy answer.
2. **Code verifies every yes.** The quote must be the caller's own words, must contain a cue for that tactic (an OTP/PIN word for an OTP request, a payment word for a money request, and so on), and must not be negated ("we never ask for OTP", "don't share details on the phone").
3. **The verdict is a fixed rule over verified evidence**: a request for money, OTP/PIN, card details or a remote-access app, or a threat, makes it SCAM; secrecy, impersonation or a link alone make it SUSPICIOUS; nothing verified is SAFE.
4. **Evon explains** the verdict in one sentence, writes the Telugu warning (from the verdict and tactics only, never the transcript) and drafts the complaint. Defender signals are shown alongside for the listener's family.

**v3.1 changes** after the first v3 run (set A 14/14, held-out set B 2/6):

- Evon sometimes paraphrases a word of its quote (మీ → "your"), and the exact-match check threw away correct evidence (SIM and QR scams missed). A quote now passes if at least 60% of its words are in one caller line.
- The money cue matched "పంప" (send), so "I sent the wedding card" counted, and it matched any rupee amount, so cash for a gas delivery counted. It now requires a *remote* payment word (transfer, UPI, online, fee, fine, …).

Sets: **A** = the 14 v2 calls (rules tuned on these), **B** = 6 calls seen once in v3 (no longer held out), **C** = 4 new calls written before this run.

In [ ]:
# 8. Test calls: (kind, name, set, transcript), Telugu-English mix as Prisma is expected to return it
CALLS = [
 ("scam", "digital_arrest", "A", '''Caller: హలో, నేను Mumbai Cyber Crime police station నుంచి Inspector Sharma మాట్లాడుతున్నాను.
Caller: మీ ఆధార్ నంబర్‌తో ఒక parcel లో drugs దొరికాయి. మీ మీద FIR register అయింది.
User: అయ్యో, నాకు ఏమీ తెలియదు సార్.
Caller: మీరు ఇప్పుడే video call లో ఉండాలి, ఈ విషయం ఎవరికీ చెప్పకూడదు, లేకపోతే arrest warrant issue చేస్తాం.
Caller: మీ savings మొత్తం RBI safe account కి transfer చేయండి, investigation తర్వాత return చేస్తాం.'''),
 ("scam", "kyc_otp", "A", '''Caller: నమస్తే అమ్మా, నేను SBI head office నుంచి మాట్లాడుతున్నాను.
Caller: మీ KYC expire అయింది, ఈరోజు update చేయకపోతే మీ account block అవుతుంది.
User: అవునా, ఏం చేయాలి?
Caller: ఇప్పుడు మీ phone కి ఒక OTP వస్తుంది, అది నాకు చెప్పండి, నేను ఇక్కడే update చేస్తాను.'''),
 ("scam", "power_cut_upi", "A", '''Caller: TSSPDCL electricity office నుంచి call చేస్తున్నాం.
Caller: మీ last month bill pending ఉంది, ఈ రాత్రి 9:30 కి మీ power connection cut చేస్తాం.
User: నేను bill కట్టేశాను కదా.
Caller: Update కాలేదు సార్. వెంటనే ఈ number కి UPI లో 1,250 rupees pay చేయండి, లేకపోతే connection cut అవుతుంది.'''),
 ("scam", "grandson_emergency", "A", '''Caller: నానమ్మా, నేను రవి. నాకు accident అయింది, hospital లో ఉన్నాను.
User: అయ్యో రవి, ఏమైంది? నీ గొంతు వేరేగా ఉంది.
Caller: జలుబు నానమ్మా. అమ్మకి చెప్పకు, టెన్షన్ పడుతుంది. Doctor కి వెంటనే 20,000 కట్టాలి.
Caller: నేను పంపే number కి Google Pay లో పంపు, ప్లీజ్ ఎవరికీ చెప్పకు.'''),
 ("scam", "remote_access", "A", '''Caller: Sir, మీ bank app లో security problem వచ్చింది, మీ money risk లో ఉంది.
Caller: Play Store లో AnyDesk అనే app download చేయండి.
User: ఎందుకు?
Caller: మా technical team మీ phone check చేస్తుంది. App open చేసి అందులో వచ్చే 9 digit code నాకు చెప్పండి.'''),
 ("scam", "lottery_fee", "A", '''Caller: Congratulations సార్! KBC lucky draw లో మీ number కి 25 లక్షలు వచ్చాయి.
User: నేను ఏ lottery కొనలేదే.
Caller: ఇది Jio customers కి free draw సార్. Prize release కావాలంటే ముందు 5,000 rupees processing fee కట్టాలి.
Caller: ఈరోజే కట్టకపోతే prize next person కి వెళ్తుంది.'''),
 ("scam", "customs_parcel", "A", '''Caller: నేను FedEx courier నుంచి మాట్లాడుతున్నాను. మీ పేరు మీద Taiwan కి వెళ్లే parcel customs లో ఆపారు.
Caller: అందులో fake passports ఉన్నాయి. నేను మిమ్మల్ని customs officer కి connect చేస్తాను.
User: నేను ఏ parcel పంపలేదు.
Caller: అయితే case close చేయాలంటే 48,000 fine ఇప్పుడే online లో కట్టాలి, మీ card details చెప్పండి.'''),
 ("genuine", "card_delivery", "A", '''Caller: నమస్కారం, నేను SBI Card customer care నుంచి మాట్లాడుతున్నాను.
Caller: మీ కొత్త credit card రేపు courier లో వస్తుంది. Delivery సమయంలో ఏదైనా ID proof చూపించండి.
User: సరే, ఏమైనా OTP చెప్పాలా?
Caller: లేదు సార్, మేము ఎప్పుడూ OTP కానీ PIN కానీ అడగము. ఏదైనా సందేహం ఉంటే card వెనుక ఉన్న number కి call చేయండి.
User: సరే, thank you.'''),
 ("genuine", "fraud_alert", "A", '''Caller: నేను HDFC Bank fraud monitoring team నుంచి మాట్లాడుతున్నాను.
Caller: మీ card మీద ఇప్పుడే 8,500 rupees online transaction వచ్చింది. అది మీరు చేశారా?
User: లేదు, నేను చేయలేదు.
Caller: సరే, మీ card ని మేము temporary గా block చేశాం. కొత్త card కోసం మీ దగ్గరలోని branch కి వెళ్ళండి లేదా app లో request పెట్టండి. మేము మిమ్మల్ని OTP లేదా PIN అడగము.'''),
 ("genuine", "hospital_reminder", "A", '''Caller: నమస్కారం, Apollo hospital నుంచి call చేస్తున్నాం.
Caller: రేపు ఉదయం 10 గంటలకు Dr. Reddy గారితో మీ appointment ఉంది.
Caller: Blood test ఉంది కాబట్టి ఖాళీ కడుపుతో రండి, పాత reports కూడా తీసుకురండి.
User: సరే, వస్తాను.'''),
 ("genuine", "grandson_visit", "A", '''Caller: నానమ్మా, నేను రవి. ఎలా ఉన్నావు?
User: బాగున్నాను రా, నువ్వు?
Caller: బాగున్నా. ఆదివారం అమ్మతో కలిసి ఇంటికి వస్తున్నాం. నీకు ఏమైనా తీసుకురావాలా?
User: ఏమీ వద్దు, మీరు రండి చాలు.'''),
 ("genuine", "power_maintenance", "A", '''Caller: TSSPDCL నుంచి సమాచారం.
Caller: రేపు line maintenance వల్ల ఉదయం 10 నుంచి మధ్యాహ్నం 2 వరకు మీ area లో power ఉండదు.
Caller: మీరు ఎలాంటి payment చేయాల్సిన అవసరం లేదు. ఇది సమాచారం మాత్రమే.'''),
 ("genuine", "pension_life_certificate", "A", '''Caller: నమస్కారం సార్, నేను Canara Bank, Ameerpet branch నుంచి మాట్లాడుతున్నాను.
Caller: మీ pension కోసం life certificate November 30 లోపు submit చేయాలి.
User: ఎలా చేయాలి?
Caller: Aadhaar card తీసుకుని branch కి రండి, లేదా Jeevan Pramaan app లో మీరే చేయవచ్చు. Phone లో ఏ details చెప్పకండి.'''),
 ("genuine", "cash_on_delivery", "A", '''Caller: Hello అమ్మా, Amazon delivery. మీ ఇంటి gate దగ్గర ఉన్నాను.
Caller: మీ order cash on delivery, 450 rupees అవుతుంది.
User: సరే, కిందకి వస్తున్నా.
Caller: సరే అమ్మా, wait చేస్తాను.'''),
 ("scam", "tax_refund_link", "B", '''Caller: Income Tax department నుంచి call చేస్తున్నాం సార్. మీకు 18,400 refund pending ఉంది.
Caller: మీ WhatsApp కి ఒక link పంపుతాను, అది open చేసి మీ bank account number, IFSC enter చేయండి.
User: సరే, చూస్తాను.
Caller: ఈరోజు రాత్రి లోపు చేయకపోతే refund cancel అవుతుంది.'''),
 ("scam", "sim_deactivation", "B", '''Caller: Airtel నుంచి మాట్లాడుతున్నాం. మీ SIM KYC verify కాలేదు, 2 గంటల్లో మీ number deactivate అవుతుంది.
User: అయ్యో, ఎలా ఆపాలి?
Caller: మీకు ఇప్పుడు ఒక SMS వస్తుంది, అందులో ఉన్న 6 digit code నాకు చదివి చెప్పండి.'''),
 ("scam", "army_qr", "B", '''Caller: నేను Army officer ని, మీరు OLX లో పెట్టిన sofa కొంటాను. 12,000 ఇస్తాను.
User: సరే, ఎలా పంపుతారు?
Caller: నేను ఒక QR code పంపుతాను, అది మీ PhonePe లో scan చేసి మీ UPI PIN enter చేయండి, డబ్బు మీ account లో పడుతుంది.'''),
 ("genuine", "fd_maturity", "B", '''Caller: నమస్కారం సార్, Union Bank Kukatpally branch నుంచి.
Caller: మీ fixed deposit వచ్చే వారం mature అవుతుంది. Renew చేయాలంటే passbook తీసుకుని branch కి రండి.
User: సరే, సోమవారం వస్తాను.'''),
 ("genuine", "gas_delivery", "B", '''Caller: Bharat Gas agency నుంచి. మీ cylinder booking confirm అయింది, రేపు ఉదయం delivery వస్తుంది.
Caller: Delivery boy కి 905 rupees cash ఇవ్వండి, bill కూడా ఇస్తాడు.
User: సరే, thank you.'''),
 ("genuine", "wedding_reminder", "B", '''Caller: హలో పెద్దనాన్నా, నేను సుధ. వచ్చే శుక్రవారం మా అమ్మాయి పెళ్ళి, మీరు తప్పకుండా రావాలి.
User: తప్పకుండా వస్తాను అమ్మా.
Caller: Card ఇంటికి పంపాను, ఎవరినైనా తోడు తీసుకుని రండి.'''),
 ("scam", "redelivery_fee_link", "C", '''Caller: India Post నుంచి మాట్లాడుతున్నాం. మీ parcel address సరిగా లేదు, delivery ఆగిపోయింది.
User: అవునా?
Caller: నేను SMS లో ఒక link పంపుతాను, అందులో 25 rupees re-delivery fee pay చేయండి, లేకపోతే parcel return అవుతుంది.'''),
 ("scam", "meter_apk", "C", '''Caller: Electricity department నుంచి. మీ meter update కాలేదు, ఈ రోజు రాత్రి power cut అవుతుంది.
User: ఏం చేయాలి సార్?
Caller: నేను WhatsApp లో ఒక APK file పంపుతాను, అది install చేసి మీ consumer number enter చేయండి.'''),
 ("genuine", "pharmacy_pickup", "C", '''Caller: నమస్కారం, Apollo Pharmacy నుంచి. మీరు order చేసిన మందులు ready గా ఉన్నాయి.
Caller: Shop కి వచ్చి తీసుకోండి, 640 rupees అవుతుంది, అప్పుడే కట్టండి.
User: సరే, సాయంత్రం వస్తాను.'''),
 ("genuine", "pension_credited", "C", '''Caller: SBI branch నుంచి సమాచారం. ఈ నెల pension మీ account లో credit అయింది.
Caller: మీరు ఏమీ చేయాల్సిన అవసరం లేదు. ఎవరైనా OTP అడిగితే చెప్పకండి.
User: సరే, thank you.'''),
]
print(len(CALLS), "calls:", sum(c[0] == "scam" for c in CALLS), "scam,", sum(c[0] == "genuine" for c in CALLS), "genuine;",
      {s: sum(c[2] == s for c in CALLS) for s in "ABC"})

In [ ]:
# 9. Role prompts (v3)
SYS = ("You analyse phone-call transcripts for Kavach, which helps elderly Telugu speakers. "
       "Most calls people receive are genuine. Judge only from what the transcript actually says, and quote lines exactly.")

TACTICS = {
 "money_or_fee_request":          "the caller asks the listener to pay, send or transfer money, a fee or a fine",
 "otp_pin_password_request":      "the caller asks the listener to tell them an OTP, PIN, password or a code from an SMS or app",
 "card_or_bank_details_request":  "the caller asks for card number, account number, IFSC, CVV or other bank details",
 "remote_access_app":             "the caller asks the listener to install or open an app like AnyDesk or TeamViewer, or to scan a QR code",
 "arrest_or_legal_threat":        "the caller threatens arrest, a police case, a warrant or legal action",
 "account_or_service_threat":     "the caller threatens to block an account, cut power, deactivate a SIM or cancel something unless the listener acts now",
 "secrecy_demand":                "the caller tells the listener not to tell family, the bank or anyone else",
 "impersonation_with_pressure":   "the caller claims to be police, customs, a court, RBI, tax or other officials and pushes the listener to act immediately",
 "suspicious_link":               "the caller sends or asks the listener to open a link"}
REQUEST_OR_THREAT = {"money_or_fee_request", "otp_pin_password_request", "card_or_bank_details_request",
                     "remote_access_app", "arrest_or_legal_threat", "account_or_service_threat"}

PRO_SCHEMA = {"type": "object", "properties": {t: {"type": "object", "properties": {
    "present": {"type": "boolean"}, "quote": {"type": "string"}}, "required": ["present", "quote"]} for t in TACTICS},
    "required": list(TACTICS)}
DEF_SCHEMA = {"type": "object", "properties": {
    "benign_signals": {"type": "array", "items": {"type": "object", "properties": {
        "signal": {"type": "string"}, "quote": {"type": "string"}}, "required": ["signal", "quote"]}}},
    "required": ["benign_signals"]}
REASON_SCHEMA = {"type": "object", "properties": {"reason": {"type": "string"}}, "required": ["reason"]}

def prosecutor(call):
    checklist = "\n".join(f"- {t}: {d}" for t, d in TACTICS.items())
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nFor each item, answer whether the CALLER does it in this call:\n{checklist}\n\n"
        "present=true only if a caller line does it; then copy that line exactly into quote. Otherwise present=false and quote=\"\". "
        "Saying they will never ask for OTP/PIN, telling the listener not to share details on the phone, pointing to the official number, "
        "branch or app, and cash payment on delivery of an order are all present=false. Most calls have every item false."}]

def defender(call):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nList the signs that this call is genuine, each with an exact caller quote: for example the caller only gives "
        "information, says they will not ask for OTP/PIN, sends the listener to an official number, branch or app, or the call is about "
        "something the listener expects. Use an empty list if there are none."}]

def explainer(call, verdict, evidence, benign):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nVerdict: {verdict}\nVerified scam evidence: {json.dumps(evidence, ensure_ascii=False)}\n"
        f"Genuine signals: {json.dumps(benign, ensure_ascii=False)}\n\nIn one English sentence, explain this verdict to the listener's family."}]

TACTIC_TE = {"money_or_fee_request": "డబ్బులు అడుగుతున్నారు", "otp_pin_password_request": "OTP లేదా PIN అడుగుతున్నారు",
             "card_or_bank_details_request": "card లేదా bank వివరాలు అడుగుతున్నారు", "remote_access_app": "phone లో app వేయమంటున్నారు",
             "arrest_or_legal_threat": "arrest చేస్తామని భయపెడుతున్నారు", "account_or_service_threat": "account లేదా connection ఆపేస్తామని భయపెడుతున్నారు",
             "secrecy_demand": "ఎవరికీ చెప్పొద్దంటున్నారు", "impersonation_with_pressure": "అధికారులమని చెప్పి తొందర పెడుతున్నారు",
             "suspicious_link": "link పంపుతున్నారు"}

def warning(verdict, tactics):
    hints = "; ".join(TACTIC_TE[t] for t in tactics if t in TACTIC_TE) or "అనుమానంగా ఉంది"
    return [{"role": "system", "content": "You write one short spoken warning in simple Telugu (Telugu script) for an elderly listener. "
             "English words common in Telugu speech (OTP, bank, app, phone) may stay in English. Never repeat what the caller said."},
            {"role": "user", "content": "Verdict: SCAM. Signs: OTP లేదా PIN అడుగుతున్నారు"},
            {"role": "assistant", "content": "అమ్మా, ఇది మోసం కాల్. OTP ఎవరికీ చెప్పకండి, వెంటనే ఫోన్ పెట్టేయండి."},
            {"role": "user", "content": f"Verdict: {verdict}. Signs: {hints}"}]

def reporter(call, verdict, tactics):
    return [{"role": "system", "content": SYS}, {"role": "user", "content":
        f"Transcript:\n{call}\n\nVerdict: {verdict}; tactics: {', '.join(tactics)}.\n"
        "Draft a short complaint the family can file on the 1930 helpline / cybercrime.gov.in: what the caller claimed, what they demanded, and the tactics. "
        "English, at most 80 words, then the same summary in Telugu."}]

In [ ]:
# 10. Verification in code: the quote must be the caller's words, carry a cue for its tactic, and not be negated
def _norm(s):
    return re.sub(r"[^\wఀ-౿₹]+", " ", s.lower()).strip()

def caller_text(call):
    return " ".join(l.split(":", 1)[1] for l in call.splitlines() if l.startswith("Caller:"))

def in_caller_lines(quote, call):
    # Evon sometimes paraphrases a word or two (మీ -> "your"), so a quote passes if at least 60% of its words
    # appear in one caller line; user lines never count
    words = _norm(re.sub(r"^\s*caller\s*:\s*", "", quote, flags=re.I).replace("...", " ").replace("…", " ")).split()
    if len(words) < 2:
        return False
    for line in call.splitlines():
        if line.startswith("Caller:"):
            lw = set(_norm(line.split(":", 1)[1]).split())
            if sum(w in lw for w in words) / len(words) >= 0.6:
                return True
    return False

CUES = {
 # remote payment only: cash handed over in person for something ordered is not a scam request
 "money_or_fee_request":         r"transfer|upi|google pay|gpay|phonepe|paytm|online|\bfee\b|\bfine\b|processing|safe account|(number|account) కి .*(పంప|pay|కట్ట)",
 "otp_pin_password_request":     r"otp|\bpin\b|password|cvv|code|కోడ్",
 "card_or_bank_details_request": r"card details|card number|account number|ifsc|cvv|details|వివరాలు",
 "remote_access_app":            r"anydesk|teamviewer|quicksupport|download|install|apk|\bqr\b|scan",
 "arrest_or_legal_threat":       r"arrest|warrant|\bfir\b|police|jail|case|court|legal",
 "account_or_service_threat":    r"block|cut|deactivat|suspend|cancel|close",
 "secrecy_demand":               r"ఎవరికీ చెప్ప|చెప్పకు|చెప్పకూడదు|don t tell|do not tell|secret",
 "impersonation_with_pressure":  r"police|officer|inspector|customs|cbi|\bed\b|rbi|court|income tax|government|head office|cyber",
 "suspicious_link":              r"link|http|www|లింక్"}
NEGATED = r"never|అడగము|అడగరు|అడగం|అవసరం లేదు|చెప్పకండి|share చేయకండి|don t share|cash on delivery"

def verify(tactic, quote, call):
    if not in_caller_lines(quote, call):
        return "not in caller lines"
    q = _norm(quote)
    if not re.search(CUES[tactic], q):
        return "no cue for tactic"
    if tactic != "secrecy_demand" and re.search(NEGATED, q):
        return "negated"
    if tactic == "secrecy_demand" and re.search(r"otp|\bpin\b|details|వివరాలు", q):
        return "protective advice"   # "don't tell anyone your OTP" protects the listener
    return "ok"

def decide(evidence):
    tactics = sorted({e["tactic"] for e in evidence})
    if REQUEST_OR_THREAT & set(tactics):
        return "SCAM", tactics
    return ("SUSPICIOUS", tactics) if tactics else ("SAFE", [])

def telugu_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return round(sum('ఀ' <= c <= '౿' for c in letters) / max(len(letters), 1), 2)

def echoes(text, call, n=3):
    grams = lambda s: {tuple(w[i:i + n]) for w in [_norm(s).split()] for i in range(len(w) - n + 1)}
    return bool(grams(text) & grams(caller_text(call)))

def parse(r):
    try:
        return json.loads(r["content"])
    except json.JSONDecodeError:
        return None

In [ ]:
# 11. Run all calls
results = []
for kind, name, cset, call in CALLS:
    t = time.time()
    with ThreadPoolExecutor(2) as ex:
        pro_r, dfn_r = ex.map(lambda a: chat(a[0], max_tokens=600, schema=a[1], temperature=0.1),
                              [(prosecutor(call), PRO_SCHEMA), (defender(call), DEF_SCHEMA)])
    pro = parse(pro_r) or {}
    dfn = parse(dfn_r) or {"benign_signals": []}
    claimed = [{"tactic": k, "quote": v.get("quote", "")} for k, v in pro.items() if k in TACTICS and v.get("present")]
    checks = [dict(e, check=verify(e["tactic"], e["quote"], call)) for e in claimed]
    evidence = [{"tactic": e["tactic"], "quote": e["quote"]} for e in checks if e["check"] == "ok"]
    benign = [b for b in dfn["benign_signals"] if in_caller_lines(b["quote"], call)]
    verdict, tactics = decide(evidence)
    t_verdict = time.time() - t
    reason = (parse(chat(explainer(call, verdict, evidence, benign), max_tokens=150, schema=REASON_SCHEMA, temperature=0.1)) or {}).get("reason", "")
    flagged = verdict != "SAFE"
    wn = chat(warning(verdict, tactics), max_tokens=120, temperature=0.3)["content"].strip() if flagged else ""
    rp = chat(reporter(call, verdict, tactics), max_tokens=500)["content"].strip() if verdict == "SCAM" else ""
    results.append({
        "kind": kind, "name": name, "set": cset, "verdict": verdict, "correct": flagged == (kind == "scam"),
        "tactics": tactics, "reason": reason, "seconds_to_verdict": round(t_verdict, 1),
        "claimed": checks, "evidence": evidence, "benign": benign,
        "warning_te": wn, "warning_telugu_ratio": telugu_ratio(wn) if wn else None,
        "warning_echoes_caller": echoes(wn, call) if wn else None, "report": rp})
    r = results[-1]
    rejected = [f"{e['tactic']}:{e['check']}" for e in checks if e["check"] != "ok"]
    print(f"{'OK ' if r['correct'] else 'BAD'} {cset} {kind:7s} {name:24s} {verdict:10s} {r['seconds_to_verdict']:5.1f}s {tactics} rejected={rejected}")
    if wn:
        print("    ", wn)
note("vram_after_roles", gpu_mem())

In [ ]:
# 12. Summary, per set
def score(rs):
    s = [r for r in rs if r["kind"] == "scam"]; g = [r for r in rs if r["kind"] == "genuine"]
    return {"scams_flagged": f"{sum(r['correct'] for r in s)}/{len(s)}", "genuine_left_alone": f"{sum(r['correct'] for r in g)}/{len(g)}"}
warned = [r for r in results if r["warning_te"]]
summary = {"set_A_tuned_on": score([r for r in results if r["set"] == "A"]),
           "set_B_seen_in_v3": score([r for r in results if r["set"] == "B"]),
           "set_C_fresh": score([r for r in results if r["set"] == "C"]),
           "median_seconds_to_verdict": sorted(r["seconds_to_verdict"] for r in results)[len(results) // 2],
           "max_seconds_to_verdict": max(r["seconds_to_verdict"] for r in results),
           "warnings_mostly_telugu": f"{sum((r['warning_telugu_ratio'] or 0) >= 0.6 for r in warned)}/{len(warned)}",
           "warnings_echoing_caller": f"{sum(bool(r['warning_echoes_caller']) for r in warned)}/{len(warned)}",
           "claims_rejected_by_code": sum(e["check"] != "ok" for r in results for e in r["claimed"]),
           "claims_accepted": sum(len(r["evidence"]) for r in results)}
note("summary", summary)

In [ ]:
# 13. Write the report and stop the server
REPORT["results"] = results
REPORT["finished"] = time.strftime("%Y-%m-%d %H:%M:%S")
Path("/kaggle/working/evon_t4_report_v31.json").write_text(json.dumps(REPORT, ensure_ascii=False, indent=2), encoding="utf-8")

lines = [f"# Kavach v3.1 on Evon v3.3 Q4_K_M, Kaggle 2x T4 ({REPORT['finished']})", "",
         "```", json.dumps(REPORT["summary"], indent=2, ensure_ascii=False), "```", "",
         "| | set | call | verdict | s | verified tactics | warning |", "|---|---|---|---|---|---|---|"]
for r in results:
    w = f"{r['warning_te']} ({r['warning_telugu_ratio']:.0%}{', ECHO' if r['warning_echoes_caller'] else ''})" if r["warning_te"] else "silent"
    lines.append(f"| {'OK' if r['correct'] else '**WRONG**'} | {r['set']} | {r['kind']}/{r['name']} | {r['verdict']} | {r['seconds_to_verdict']} | "
                 f"{', '.join(r['tactics'])} | {w} |")
for r in results:
    lines += ["", f"### {r['set']} {r['kind']}/{r['name']}: {r['verdict']}", "", f"Reason: {r['reason']}", ""]
    lines += [f"- claimed {e['tactic']} ({e['check']}): {e['quote']}" for e in r["claimed"]]
    lines += [f"- genuine signal: {b['signal']}: {b['quote']}" for b in r["benign"]]
    if r["report"]:
        lines += ["", "Report:", "", r["report"]]
Path("/kaggle/working/evon_t4_report_v31.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:35]))

server.terminate()
print("\nwrote /kaggle/working/evon_t4_report_v31.md and .json")